In [ ]:
from IPython.display import display
from time import sleep
from typing import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver



#1. 声明状态
class OverAllState(TypedDict):
    username:str
    age:int

#2. 声明节点
def node_a(state:OverAllState) -> OverAllState:
    username = interrupt("请输入您的姓名:")
    return {
        "username":username
    }

def node_b(state:OverAllState) -> OverAllState:
    sleep(1)
    age = interrupt("请输入您的年龄:")
    return {
        "age":age
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a",node_a)
builder.add_node("node_b",node_b)
builder.add_edge(START,"node_a")
builder.add_edge(START,"node_b")
builder.add_edge("node_a",END)
builder.add_edge("node_b",END)

#4. 添加检查点后端
checkpointer=InMemorySaver()
graph=builder.compile(checkpointer=checkpointer)
display(graph)
#5. 执行第一个图 => 触发中断
config={"configurable":{"thread_id":"123"}}
interrupt_res=graph.invoke({},config=config)
print(interrupt_res)


In [ ]:
#6. 恢复图的中断执行
resume_map={}
for i in interrupt_res['__interrupt__']:
    user_input=input(f"{i.value}")
    if "年龄" in i.value:
        resume_map[i.id]=int(user_input)
    else:
        resume_map[i.id]=user_input

resumed_res=graph.invoke(Command(resume=resume_map),config=config)
print(resumed_res)